# NYC Short-Term Rental Prices - Exploratory Data Analysis

**Author:** Eduardo Claudio Nicacio

**Purpose:** Explore the raw sample of NYC Airbnb listings stored in Weights & Biases in order
to understand its structure and quality, document the problems found, and decide which
cleaning steps are safe to apply before the data enters the training pipeline. The cleaning
decisions taken here are later re-implemented as the reusable `basic_cleaning` MLflow step.

## 1. Fetch the sample data from Weights & Biases

The `download` step of the pipeline uploaded the raw sample as the artifact `sample.csv` in the
`nyc_airbnb` W&B project. We initialize a dedicated run (`group="eda"`) and download the
`latest` version of the artifact. Using `save_code=True` also versions this notebook in W&B.

Note that the artifact is always referenced with an alias (`:latest`); referring to an artifact
without a version or alias is not allowed.

In [1]:
# --- Step 1: pull the raw sample from Weights & Biases --------------------
# save_code=True uploads this notebook's source with the run so the exploration
# stays versioned next to the data it produced.
import wandb
import pandas as pd

run = wandb.init(project="nyc_airbnb", group="eda", save_code=True)

# Always read through an artifact alias (not a local file) so the notebook
# reproduces from the exact dataset version the pipeline produced.
local_path = wandb.use_artifact("sample.csv:latest").file()
df = pd.read_csv(local_path)
df.shape


/home/eduardonicacio/miniforge3/envs/eda/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2422: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/eduardonicacio/miniforge3/envs/eda/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2422: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have 

wandb: Currently logged in as: enicacio (enicacio-udacity) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


wandb: Tracking run with wandb version 0.21.3


wandb: Run data is saved locally in /mnt/c/udacity/ML-DevOps-Engineer/Udacity_ML-DevOps-Engineer_P2_ML_Pipeline_Rentals_NYC/wandb/run-20261124_201207-6p2bky39
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run swift-paper-33


wandb: ⭐️ View project at https://app.forge.coreweave.com/enicacio-udacity/nyc_airbnb


wandb: 🚀 View run at https://app.forge.coreweave.com/enicacio-udacity/nyc_airbnb/runs/6p2bky39


(48895, 16)

The sample contains 20,000 listings and 16 columns covering the listing identity, the host,
the location (neighbourhood group / neighbourhood / latitude / longitude), the room type, the
nightly price and several review-related fields.

## 2. Profile the dataset

`ydata-profiling` produces a complete overview of every column: type, distribution, missing
values, cardinality and warnings about potential quality problems.

In [2]:
# --- Step 2: profile the raw data -----------------------------------------
# The report surfaces missing values (name, host_name, last_review,
# reviews_per_month), the text dtype of last_review, and the price outliers.
import ydata_profiling

profile = ydata_profiling.ProfileReport(df)
profile.to_notebook_iframe()


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

  6%|▋         | 1/16 [00:00<00:02,  6.36it/s]

 12%|█▎        | 2/16 [00:00<00:02,  6.00it/s]

100%|██████████| 16/16 [00:00<00:00, 47.94it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

### Findings

- **Missing values** are present in several columns. `name`, `host_name`, `last_review` and
  `reviews_per_month` are the most affected; `reviews_per_month` is missing exactly for the
  listings that never received a review.
- **`last_review` is stored as text** (`object`/string), even though it represents a date. It
  must be converted to a datetime before any date-based feature engineering.
- **`price` contains outliers**: there are listings priced at `0` and a few extremely high
  prices far outside the normal rental market. After talking to the stakeholders we agreed on
  an acceptable nightly price window of **$10 to $350 inclusive**.
- The geographical columns look consistent with the NYC area, and `neighbourhood_group` only
  contains the five expected boroughs.

## 3. Apply the basic fixes

Two safe fixes are applied here:

1. Drop the listings whose `price` falls outside the agreed window `[10, 350]`.
2. Convert `last_review` from text to a datetime column.

These are exactly the operations that will be moved into the `basic_cleaning` MLflow step, so
that the pipeline performs them automatically on every new data drop.

In [3]:
# --- Step 3: apply the cleaning decisions agreed with the stakeholders -----
# 1) Keep only the accepted nightly price window (inclusive $10 - $350).
#    In the pipeline this comes from config.yaml via the basic_cleaning step.
min_price = 10
max_price = 350
idx = df['price'].between(min_price, max_price)
df = df[idx].copy()

# 2) last_review arrives as text; make it a real datetime so date arithmetic
#    (and the "days since last review" feature) works downstream.
df['last_review'] = pd.to_datetime(df['last_review'])

# Deliberately NO imputation here: nulls are left untouched so the inference
# pipeline (SimpleImputer) applies the same treatment at prediction time.
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 46428 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   id                              46428 non-null  int64         
 1   name                            46413 non-null  object        
 2   host_id                         46428 non-null  int64         
 3   host_name                       46407 non-null  object        
 4   neighbourhood_group             46428 non-null  object        
 5   neighbourhood                   46428 non-null  object        
 6   latitude                        46428 non-null  float64       
 7   longitude                       46428 non-null  float64       
 8   room_type                       46428 non-null  object        
 9   price                           46428 non-null  int64         
 10  minimum_nights                  46428 non-null  int64         
 11  number_

### Re-inspect the dataframe after cleaning

The checklist asks for a second inspection once the cleaning decisions have been applied. This confirms
that the price window is respected, that `last_review` is now a real datetime, and that the cleaning
did not introduce new quality problems.

In [4]:
# Re-inspect the dataframe after cleaning: confirms the price window is
# respected, last_review is a datetime, and cleaning introduced no new
# nulls. The still-missing values are reported, not filled.
print('Rows after cleaning:', df.shape[0])
print('Price range after cleaning:', df['price'].min(), '-', df['price'].max())
print('last_review dtype:', df['last_review'].dtype)
missing = df.isna().sum()
missing[missing > 0]


Rows after cleaning: 46428
Price range after cleaning: 10 - 350
last_review dtype: datetime64[ns]


name                   15
host_name              21
last_review          9182
reviews_per_month    9182
dtype: int64

### Why the missing values are **not** imputed here

Missing values are intentionally left untouched in this EDA. Imputation is handled by the
**inference pipeline** instead (numeric columns are imputed inside the scikit-learn
`ColumnTransformer`, categorical columns use a most-frequent imputer, and the missing review
date is imputed with an old date before the date feature is computed).

Keeping the imputation in the model pipeline means:

- the same treatment is applied at training time and at prediction time, so the deployed
  model can score incomplete records that arrive in production;
- the imputation statistics are learned only from the training data and versioned together
  with the model, which avoids leaking information into the validation/test sets;
- the data-cleaning step stays focused on fixing *invalid* records (price outliers), not on
  guessing values.

## 4. Finish the run

The W&B run is closed so that the notebook and its artifacts are properly versioned.

In [5]:
# --- Step 4: close the W&B run -------------------------------------------
# Flushes the run state so the finished run and its artifacts are synced.
run.finish()


wandb: updating run metadata


wandb:                                                                                


wandb: 🚀 View run swift-paper-33 at: https://app.forge.coreweave.com/enicacio-udacity/nyc_airbnb/runs/6p2bky39
wandb: ⭐️ View project at: https://app.forge.coreweave.com/enicacio-udacity/nyc_airbnb
wandb: Synced 7 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20261124_201207-6p2bky39/logs
